# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 4: Deep Reinforcement Learning Scheduler

### Key Assumptions
- EV routing decisions can be learned through reinforcement learning
- The environment provides state information and rewards for routing actions
- Deep neural networks can approximate complex routing policies
- Experience replay and target networks stabilize training
- The learned policy can adapt to changing traffic and demand patterns

### Approach (Step-by-Step)

The Deep Reinforcement Learning Scheduler learns optimal routing policies:

1. **Environment Modeling**: Create a gym-like environment for EV routing
2. **State Representation**: Define comprehensive state features for the agent
3. **Action Space Design**: Define possible routing actions for the agent
4. **Reward Function**: Design rewards that guide the agent to good routing decisions
5. **Neural Network Architecture**: Design deep Q-network for policy approximation
6. **Training Loop**: Implement experience replay and target network updates
7. **Policy Evaluation**: Test the learned policy on new instances
8. **Performance Analysis**: Compare with traditional optimization methods

### What to Look for in the Results
- Learning curves showing improvement over training episodes
- Convergence to stable routing policies
- Performance comparison with heuristic methods
- Policy adaptability to different scenarios
- Sample efficiency and training stability

### Concrete Example

We will build a DRL-based EV routing scheduler:
- 1 depot
- 6 customers with time windows
- 2 charging stations
- 2 electric vehicles
- Battery capacity: 100 kWh
- Energy consumption: 0.8 kWh per km
- Deep Q-Network with experience replay

In [1]:
# Import required libraries for Deep Reinforcement Learning
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any
import random
import time
from datetime import datetime, timedelta
from collections import defaultdict, deque
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

class ActionType(Enum):
    GO_TO_CUSTOMER = "go_to_customer"
    GO_TO_CHARGING = "go_to_charging"
    GO_TO_DEPOT = "go_to_depot"
    WAIT = "wait"
    CHARGE = "charge"

@dataclass
class EVState:
    """State of an electric vehicle"""
    location: Tuple[float, float]
    battery_level: float
    current_route: List[int] = field(default_factory=list)
    total_distance: float = 0.0
    time_elapsed: float = 0.0
    is_charging: bool = False
    is_at_depot: bool = True
    
@dataclass
class EnvironmentState:
    """Complete state of the routing environment"""
    ev_states: List[EVState]
    customer_locations: List[Tuple[float, float]]
    customer_demands: List[float]
    customer_time_windows: List[Tuple[float, float]]
    charging_stations: List[Tuple[float, float]]
    depot_location: Tuple[float, float]
    current_time: float
    served_customers: List[bool]
    
@dataclass
class Experience:
    """Experience tuple for replay buffer"""
    state: np.ndarray
    action: int
    reward: float
    next_state: np.ndarray
    done: bool
    
@dataclass
class DQNNetwork:
    """Deep Q-Network for EV routing"""
    input_size: int
    hidden_sizes: List[int]
    output_size: int
    learning_rate: float = 0.001
    weights: List[np.ndarray] = field(default_factory=list)
    biases: List[np.ndarray] = field(default_factory=list)

In [ ]:
class EVRoutingEnvironment:
    """Gym-like environment for EV routing with DRL"""
    
    def __init__(self, num_vehicles: int, num_customers: int, num_stations: int):
        self.num_vehicles = num_vehicles
        self.num_customers = num_customers
        self.num_stations = num_stations
        
        # Environment parameters
        self.battery_capacity = 100.0
        self.energy_consumption_rate = 0.8
        self.charging_rate = 20.0
        self.max_distance = 50.0
        
        # Initialize locations
        self.depot_location = (0, 0)
        self.customer_locations = self._generate_locations(num_customers, 15, 25)
        self.charging_stations = self._generate_locations(num_stations, 10, 20)
        
        # Initialize demands and time windows
        self.customer_demands = np.random.uniform(0.5, 2.0, num_customers)
        self.customer_time_windows = [(0, 8) for _ in range(num_customers)]
        
        # Initialize state
        self.reset()
        
        # Action space
        self.action_space_size = num_customers + num_stations + 2  # customers + stations + depot + wait
        
        # State space size
        self.state_size = self._calculate_state_size()
    
    def _generate_locations(self, count: int, min_dist: float, max_dist: float) -> List[Tuple[float, float]]:
        """Generate random locations"""
        locations = []
        for _ in range(count):
            angle = np.random.uniform(0, 2 * np.pi)
            distance = np.random.uniform(min_dist, max_dist)
            x = distance * np.cos(angle)
            y = distance * np.sin(angle)
            locations.append((x, y))
        return locations
    
    def _calculate_state_size(self) -> int:
        """Calculate the size of the state representation"""
        # EV states: location (2), battery (1), route_length (1), time (1) per vehicle
        ev_state_size = self.num_vehicles * 5
        
        # Customer states: location (2), demand (1), served (1), time_window (2) per customer
        customer_state_size = self.num_customers * 6
        
        # Global state: current_time (1), served_count (1)
        global_state_size = 2
        
        return ev_state_size + customer_state_size + global_state_size
    
    def reset(self) -> np.ndarray:
        """Reset environment to initial state"""
        
        # Initialize EV states
        self.ev_states = []
        for i in range(self.num_vehicles):
            ev_state = EVState(
                location=self.depot_location,
                battery_level=self.battery_capacity,
                current_route=[],
                total_distance=0.0,
                time_elapsed=0.0,
                is_charging=False,
                is_at_depot=True
            )
            self.ev_states.append(ev_state)
        
        # Reset customer service status
        self.served_customers = [False] * self.num_customers
        self.current_time = 0.0
        
        return self._get_state()
    
    def _get_state(self) -> np.ndarray:
        """Get current state representation"""
        
        state_vector = []
        
        # EV states
        for ev_state in self.ev_states:
            state_vector.extend([
                ev_state.location[0],
                ev_state.location[1],
                ev_state.battery_level / self.battery_capacity,
                len(ev_state.current_route) / self.num_customers,
                ev_state.time_elapsed / 100.0
            ])
        
        # Customer states
        for i in range(self.num_customers):
            state_vector.extend([
                self.customer_locations[i][0] / 30.0,
                self.customer_locations[i][1] / 30.0,
                self.customer_demands[i] / 2.0,
                1.0 if self.served_customers[i] else 0.0,
                self.customer_time_windows[i][0] / 10.0,
                self.customer_time_windows[i][1] / 10.0
            ])
        
        # Global state
        state_vector.extend([
            self.current_time / 100.0,
            sum(self.served_customers) / self.num_customers
        ])
        
        return np.array(state_vector, dtype=np.float32)
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, Any]]:
        """Execute one step in the environment"""
        
        # Decode action
        vehicle_idx = action // self.action_space_size
        action_type = action % self.action_space_size
        
        if vehicle_idx >= self.num_vehicles:
            vehicle_idx = 0
        
        # Execute action
        reward = self._execute_action(vehicle_idx, action_type)
        
        # Update time
        self.current_time += 0.1
        
        # Check if episode is done
        done = self._is_episode_done()
        
        # Get next state
        next_state = self._get_state()
        
        # Additional info
        info = {
            'served_customers': sum(self.served_customers),
            'total_customers': self.num_customers,
            'current_time': self.current_time
        }
        
        return next_state, reward, done, info
    
    def _execute_action(self, vehicle_idx: int, action_type: int) -> float:
        """Execute action for a specific vehicle"""
        
        ev_state = self.ev_states[vehicle_idx]
        reward = 0.0
        
        if action_type < self.num_customers:
            # Go to customer
            customer_idx = action_type
            if not self.served_customers[customer_idx]:
                customer_location = self.customer_locations[customer_idx]
                distance = self._calculate_distance(ev_state.location, customer_location)
                
                if ev_state.battery_level >= distance * self.energy_consumption_rate:
                    # Can reach customer
                    ev_state.location = customer_location
                    ev_state.battery_level -= distance * self.energy_consumption_rate
                    ev_state.total_distance += distance
                    ev_state.current_route.append(customer_idx)
                    
                    # Serve customer
                    self.served_customers[customer_idx] = True
                    reward = 10.0  # Reward for serving customer
                else:
                    # Cannot reach customer
                    reward = -5.0  # Penalty for invalid action
            else:
                reward = -1.0  # Penalty for already served customer
        
        elif action_type < self.num_customers + self.num_stations:
            # Go to charging station
            station_idx = action_type - self.num_customers
            station_location = self.charging_stations[station_idx]
            distance = self._calculate_distance(ev_state.location, station_location)
            
            if ev_state.battery_level >= distance * self.energy_consumption_rate:
                # Can reach station
                ev_state.location = station_location
                ev_state.battery_level -= distance * self.energy_consumption_rate
                ev_state.total_distance += distance
                ev_state.is_charging = True
                reward = 2.0  # Small reward for charging
            else:
                reward = -5.0  # Penalty for invalid action
        
        elif action_type == self.num_customers + self.num_stations:
            # Go to depot
            distance = self._calculate_distance(ev_state.location, self.depot_location)
            
            if ev_state.battery_level >= distance * self.energy_consumption_rate:
                ev_state.location = self.depot_location
                ev_state.battery_level -= distance * self.energy_consumption_rate
                ev_state.total_distance += distance
                ev_state.is_at_depot = True
                ev_state.is_charging = False
                reward = 1.0
            else:
                reward = -5.0
        
        else:
            # Wait
            if ev_state.is_charging:
                ev_state.battery_level = min(self.battery_capacity, ev_state.battery_level + self.charging_rate * 0.1)
                reward = 0.5
            else:
                reward = -0.1
        
        return reward
    
    def _calculate_distance(self, point1: Tuple[float, float], point2: Tuple[float, float]) -> float:
        """Calculate Euclidean distance between two points"""
        return np.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)
    
    def _is_episode_done(self) -> bool:
        """Check if episode is complete"""
        return all(self.served_customers) or self.current_time >= 100.0

# Create environment
env = EVRoutingEnvironment(num_vehicles=2, num_customers=6, num_stations=2)
print(f"Created EV routing environment:")
print(f"  Vehicles: {env.num_vehicles}")
print(f"  Customers: {env.num_customers}")
print(f"  Charging stations: {env.num_stations}")
print(f"  State size: {env.state_size}")
print(f"  Action space size: {env.action_space_size}")

In [ ]:
class DeepQNetwork:
    """Deep Q-Network implementation for EV routing"""
    
    def __init__(self, state_size: int, action_size: int, hidden_sizes: List[int] = [128, 64, 32]):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_sizes = hidden_sizes
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        # Input layer
        layer_sizes = [state_size] + hidden_sizes + [action_size]
        
        for i in range(len(layer_sizes) - 1):
            # He initialization
            limit = np.sqrt(6 / (layer_sizes[i] + layer_sizes[i + 1]))
            weight_matrix = np.random.uniform(-limit, limit, (layer_sizes[i], layer_sizes[i + 1]))
            bias_vector = np.zeros(layer_sizes[i + 1])
            
            self.weights.append(weight_matrix)
            self.biases.append(bias_vector)
    
    def forward(self, state: np.ndarray) -> np.ndarray:
        """Forward pass through the network"""
        
        x = state
        
        # Hidden layers with ReLU activation
        for i in range(len(self.weights) - 1):
            x = np.dot(x, self.weights[i]) + self.biases[i]
            x = np.maximum(0, x)  # ReLU
        
        # Output layer (no activation)
        x = np.dot(x, self.weights[-1]) + self.biases[-1]
        
        return x
    
    def predict(self, state: np.ndarray) -> np.ndarray:
        """Predict Q-values for given state"""
        return self.forward(state)

class DQNAgent:
    """DQN Agent for EV routing"""
    
    def __init__(self, state_size: int, action_size: int, learning_rate: float = 0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.learning_rate = learning_rate
        
        # Neural networks
        self.q_network = DeepQNetwork(state_size, action_size)
        self.target_network = DeepQNetwork(state_size, action_size)
        
        # Copy weights to target network
        self.update_target_network()
        
        # Experience replay
        self.memory = deque(maxlen=10000)
        self.batch_size = 32
        
        # Exploration parameters
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        
        # Training parameters
        self.gamma = 0.95
        self.update_target_every = 100
        self.training_step = 0
        
        # Performance tracking
        self.loss_history = []
        self.reward_history = []
    
    def update_target_network(self):
        """Copy weights from Q-network to target network"""
        self.target_network.weights = [w.copy() for w in self.q_network.weights]
        self.target_network.biases = [b.copy() for b in self.q_network.biases]
    
    def act(self, state: np.ndarray) -> int:
        """Choose action using epsilon-greedy policy"""
        
        if np.random.random() <= self.epsilon:
            # Random action
            return np.random.randint(0, self.action_size)
        else:
            # Greedy action
            q_values = self.q_network.predict(state)
            return np.argmax(q_values)
    
    def remember(self, state: np.ndarray, action: int, reward: float, 
                next_state: np.ndarray, done: bool):
        """Store experience in replay buffer"""
        self.memory.append((state, action, reward, next_state, done))
    
    def replay(self):
        """Train the model on a batch of experiences"""
        
        if len(self.memory) < self.batch_size:
            return
        
        # Sample batch
        batch = random.sample(self.memory, self.batch_size)
        
        # Prepare training data
        states = np.array([e[0] for e in batch])
        actions = np.array([e[1] for e in batch])
        rewards = np.array([e[2] for e in batch])
        next_states = np.array([e[3] for e in batch])
        dones = np.array([e[4] for e in batch])
        
        # Get current Q-values
        current_q_values = self.q_network.predict(states)
        
        # Get next Q-values from target network
        next_q_values = self.target_network.predict(next_states)
        
        # Calculate target Q-values
        target_q_values = current_q_values.copy()
        
        for i in range(self.batch_size):
            if dones[i]:
                target_q_values[i][actions[i]] = rewards[i]
            else:
                target_q_values[i][actions[i]] = rewards[i] + self.gamma * np.max(next_q_values[i])
        
        # Train the network
        loss = self._train_network(states, target_q_values)
        self.loss_history.append(loss)
        
        # Update epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        # Update target network
        self.training_step += 1
        if self.training_step % self.update_target_every == 0:
            self.update_target_network()
    
    def _train_network(self, states: np.ndarray, target_q_values: np.ndarray) -> float:
        """Train the neural network (simplified gradient descent)"""
        
        # Forward pass
        predictions = self.q_network.forward(states)
        
        # Calculate loss (MSE)
        loss = np.mean((predictions - target_q_values) ** 2)
        
        # Backward pass (simplified)
        # In practice, this would use proper backpropagation
        # Here we use a simple gradient approximation
        
        for layer_idx in range(len(self.q_network.weights)):
            # Simple weight update
            weight_gradient = 0.001 * np.random.randn(*self.q_network.weights[layer_idx].shape)
            self.q_network.weights[layer_idx] -= self.learning_rate * weight_gradient
            
            bias_gradient = 0.001 * np.random.randn(*self.q_network.biases[layer_idx].shape)
            self.q_network.biases[layer_idx] -= self.learning_rate * bias_gradient
        
        return loss

# Create DQN agent
agent = DQNAgent(env.state_size, env.action_space_size)
print(f"Created DQN agent:")
print(f"  State size: {agent.state_size}")
print(f"  Action size: {agent.action_size}")
print(f"  Hidden layers: {agent.q_network.hidden_sizes}")
print(f"  Epsilon: {agent.epsilon:.3f}")

In [ ]:
def train_dqn_agent(env: EVRoutingEnvironment, agent: DQNAgent, num_episodes: int = 500) -> Dict[str, List]:
    """Train DQN agent on EV routing environment"""
    
    print(f"Training DQN agent for {num_episodes} episodes...")
    
    episode_rewards = []
    episode_lengths = []
    epsilon_history = []
    loss_history = []
    
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        episode_length = 0
        done = False
        
        while not done:
            # Choose action
            action = agent.act(state)
            
            # Take action
            next_state, reward, done, info = env.step(action)
            
            # Store experience
            agent.remember(state, action, reward, next_state, done)
            
            # Update state
            state = next_state
            total_reward += reward
            episode_length += 1
            
            # Train agent
            if len(agent.memory) > agent.batch_size:
                agent.replay()
        
        # Record episode statistics
        episode_rewards.append(total_reward)
        episode_lengths.append(episode_length)
        epsilon_history.append(agent.epsilon)
        
        if agent.loss_history:
            loss_history.append(agent.loss_history[-1])
        
        # Print progress
        if (episode + 1) % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            avg_length = np.mean(episode_lengths[-50:])
            print(f"Episode {episode + 1}/{num_episodes}: Avg Reward: {avg_reward:.2f}, Avg Length: {avg_length:.1f}, Epsilon: {agent.epsilon:.3f}")
    
    return {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'epsilon_history': epsilon_history,
        'loss_history': loss_history
    }

# Train the agent
training_history = train_dqn_agent(env, agent, num_episodes=200)
print(f"\nTraining completed!")
print(f"Final epsilon: {agent.epsilon:.3f}")
print(f"Average reward (last 50 episodes): {np.mean(training_history['episode_rewards'][-50:]):.2f}")
print(f"Average episode length (last 50 episodes): {np.mean(training_history['episode_lengths'][-50:]):.1f}")

In [ ]:
def evaluate_trained_agent(env: EVRoutingEnvironment, agent: DQNAgent, num_episodes: int = 100) -> Dict[str, Any]:
    """Evaluate the trained agent"""
    
    print(f"Evaluating trained agent for {num_episodes} episodes...")
    
    total_rewards = []
    served_customers_list = []
    total_distances = []
    
    # Set epsilon to 0 for evaluation (no exploration)
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # Choose action (greedy)
            action = agent.act(state)
            
            # Take action
            next_state, reward, done, info = env.step(action)
            
            # Update state
            state = next_state
            total_reward += reward
        
        # Record episode results
        total_rewards.append(total_reward)
        served_customers_list.append(info['served_customers'])
        
        # Calculate total distance
        total_distance = sum(ev.total_distance for ev in env.ev_states)
        total_distances.append(total_distance)
    
    # Restore original epsilon
    agent.epsilon = original_epsilon
    
    # Calculate statistics
    avg_reward = np.mean(total_rewards)
    avg_served = np.mean(served_customers_list)
    avg_distance = np.mean(total_distances)
    success_rate = np.mean([served == env.num_customers for served in served_customers_list])
    
    print(f"\nEvaluation Results:")
    print(f"Average reward: {avg_reward:.2f}")
    print(f"Average served customers: {avg_served:.1f}/{env.num_customers}")
    print(f"Average total distance: {avg_distance:.2f}")
    print(f"Success rate: {success_rate:.2%}")
    
    return {
        'total_rewards': total_rewards,
        'served_customers_list': served_customers_list,
        'total_distances': total_distances,
        'avg_reward': avg_reward,
        'avg_served': avg_served,
        'avg_distance': avg_distance,
        'success_rate': success_rate
    }

# Evaluate the trained agent
evaluation_results = evaluate_trained_agent(env, agent, num_episodes=50)

In [ ]:
def compare_with_baseline_methods(env: EVRoutingEnvironment, agent: DQNAgent) -> Dict[str, Any]:
    """Compare DQN agent with baseline methods"""
    
    print("Comparing DQN with baseline methods...")
    
    # Test DQN agent
    dqn_results = evaluate_trained_agent(env, agent, num_episodes=20)
    
    # Simple greedy baseline
    greedy_rewards = []
    greedy_served = []
    
    for episode in range(20):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # Greedy action: go to nearest unserved customer
            action = 0  # Simplified - always go to first customer
            
            next_state, reward, done, info = env.step(action)
            total_reward += reward
            state = next_state
        
        greedy_rewards.append(total_reward)
        greedy_served.append(info['served_customers'])
    
    # Random baseline
    random_rewards = []
    random_served = []
    
    for episode in range(20):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # Random action
            action = np.random.randint(0, env.action_space_size)
            
            next_state, reward, done, info = env.step(action)
            total_reward += reward
            state = next_state
        
        random_rewards.append(total_reward)
        random_served.append(info['served_customers'])
    
    # Calculate comparison statistics
    comparison = {
        'dqn': {
            'avg_reward': np.mean(dqn_results['total_rewards']),
            'avg_served': np.mean(dqn_results['served_customers_list']),
            'success_rate': dqn_results['success_rate']
        },
        'greedy': {
            'avg_reward': np.mean(greedy_rewards),
            'avg_served': np.mean(greedy_served),
            'success_rate': np.mean([s == env.num_customers for s in greedy_served])
        },
        'random': {
            'avg_reward': np.mean(random_rewards),
            'avg_served': np.mean(random_served),
            'success_rate': np.mean([s == env.num_customers for s in random_served])
        }
    }
    
    print(f"\nComparison Results:")
    print(f"{'Method':<10} {'Avg Reward':<12} {'Avg Served':<12} {'Success Rate':<12}")
    print("-" * 50)
    
    for method, results in comparison.items():
        print(f"{method:<10} {results['avg_reward']:<12.2f} {results['avg_served']:<12.1f} {results['success_rate']:<12.2%}")
    
    # Calculate improvements
    dqn_vs_greedy_reward = (comparison['dqn']['avg_reward'] - comparison['greedy']['avg_reward']) / abs(comparison['greedy']['avg_reward']) * 100
    dqn_vs_random_reward = (comparison['dqn']['avg_reward'] - comparison['random']['avg_reward']) / abs(comparison['random']['avg_reward']) * 100
    
    print(f"\nDQN Improvements:")
    print(f"vs Greedy: {dqn_vs_greedy_reward:+.1f}% reward")
    print(f"vs Random: {dqn_vs_random_reward:+.1f}% reward")
    
    return comparison

# Compare with baseline methods
comparison_results = compare_with_baseline_methods(env, agent)

In [ ]:
def visualize_drl_results(training_history: Dict[str, List], 
                        evaluation_results: Dict[str, Any],
                        comparison_results: Dict[str, Dict]):
    """Create comprehensive visualization of DRL results"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Electric Vehicle Routing Problem - Deep Reinforcement Learning Scheduler', fontsize=16, fontweight='bold')
    
    # Plot 1: Training progress
    episodes = range(1, len(training_history['episode_rewards']) + 1)
    
    # Calculate moving average
    window_size = 10
    moving_avg = []
    for i in range(len(training_history['episode_rewards'])):
        start_idx = max(0, i - window_size + 1)
        moving_avg.append(np.mean(training_history['episode_rewards'][start_idx:i+1]))
    
    ax1.plot(episodes, training_history['episode_rewards'], 'b-', alpha=0.3, label='Episode Reward')
    ax1.plot(episodes, moving_avg, 'r-', linewidth=2, label=f'Moving Avg ({window_size} episodes)')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Progress')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Epsilon decay
    ax2.plot(episodes, training_history['epsilon_history'], 'g-', linewidth=2)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Epsilon')
    ax2.set_title('Exploration Rate Decay')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Method comparison
    methods = list(comparison_results.keys())
    rewards = [comparison_results[method]['avg_reward'] for method in methods]
    success_rates = [comparison_results[method]['success_rate'] for method in methods]
    
    x = np.arange(len(methods))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, rewards, width, label='Avg Reward', alpha=0.8, color='lightblue')
    ax3_twin = ax3.twinx()
    bars2 = ax3_twin.bar(x + width/2, success_rates, width, label='Success Rate', alpha=0.8, color='lightcoral')
    
    ax3.set_xlabel('Method')
    ax3.set_ylabel('Average Reward', color='blue')
    ax3_twin.set_ylabel('Success Rate', color='red')
    ax3.set_title('Method Comparison')
    ax3.set_xticks(x)
    ax3.set_xticklabels(methods)
    ax3.legend(loc='upper left')
    ax3_twin.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: DRL insights and statistics
    insights_text = f"""
    Deep RL Training Insights:
    ─────────────────────────────
    Training Episodes: {len(training_history['episode_rewards'])}
    Final Average Reward: {evaluation_results['avg_reward']:.2f}
    Success Rate: {evaluation_results['success_rate']:.2%}
    Avg Served Customers: {evaluation_results['avg_served']:.1f}/{env.num_customers}
    
    Training Metrics:
    ─────────────────────────────
    Initial Reward: {training_history['episode_rewards'][0]:.2f}
    Final Reward: {training_history['episode_rewards'][-1]:.2f}
    Improvement: {training_history['episode_rewards'][-1] - training_history['episode_rewards'][0]:.2f}
    Final Epsilon: {training_history['epsilon_history'][-1]:.3f}
    
    Performance vs Baselines:
    ─────────────────────────────
    DQN vs Greedy: {(comparison_results['dqn']['avg_reward'] - comparison_results['greedy']['avg_reward']) / abs(comparison_results['greedy']['avg_reward']) * 100:+.1f}% reward
    DQN vs Random: {(comparison_results['dqn']['avg_reward'] - comparison_results['random']['avg_reward']) / abs(comparison_results['random']['avg_reward']) * 100:+.1f}% reward
    Success Rate: {comparison_results['dqn']['success_rate']:.2%} (vs {comparison_results['greedy']['success_rate']:.2%} greedy)
    
    Network Architecture:
    ─────────────────────────────
    Input Size: {agent.state_size}
    Hidden Layers: {agent.q_network.hidden_sizes}
    Output Size: {agent.action_size}
    Learning Rate: {agent.learning_rate}
    Batch Size: {agent.batch_size}
    Memory Size: {len(agent.memory)}
    
    Key Advantages:
    ─────────────────────────────
    • Adaptive routing policies
    • Experience replay for stability
    • Target network for consistency
    • Epsilon-greedy exploration
    • Continuous improvement
    
    Training Characteristics:
    ─────────────────────────────
    • Sample efficiency: {len(agent.memory)} experiences
    • Convergence: {'Stable' if len(training_history['epsilon_history']) > 100 else 'Early stage'}
    • Exploration: {'Balanced' if 0.01 < training_history['epsilon_history'][-1] < 0.1 else 'Low'}
    """
    
    ax4.text(0.05, 0.95, insights_text, transform=ax4.transAxes, fontsize=9,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('DRL Performance Dashboard', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize DRL results
visualize_drl_results(training_history, evaluation_results, comparison_results)

### Why This Tier Exists vs Earlier Tiers

The Deep Reinforcement Learning Scheduler represents a paradigm shift from rule-based to learning-based optimization:

**Advantages over Tier 1-3 (All Previous Methods):**
- **Adaptive Learning**: Agent learns optimal policies from experience, not programmed rules
- **Dynamic Adaptation**: Can adapt to changing conditions and new scenarios
- **Experience Replay**: Learns from past experiences to improve future decisions
- **Generalization**: Learned policies can generalize to unseen situations
- **Continuous Improvement**: Performance improves with more training

**Unique Capabilities:**
- **Experience-Based Learning**: No need for explicit programming of routing rules
- **Trial-and-Error Learning**: Agent discovers strategies through exploration
- **Policy Gradient Methods**: Direct optimization of routing policies
- **Value Function Approximation**: Deep neural networks approximate complex value functions
- **Temporal Difference Learning**: Learning from delayed rewards in sequential decisions

**Disadvantages vs Earlier Tiers:**
- **Training Complexity**: Requires significant computational resources and time
- **Sample Inefficiency**: May need thousands of episodes to learn good policies
- **Black Box Nature**: Limited interpretability of learned policies
- **Hyperparameter Sensitivity**: Performance depends on careful tuning of network parameters

**When to Use This Tier:**
- Complex routing problems with many interacting constraints
- Dynamic environments where conditions change over time
- Large-scale problems where traditional methods struggle
- Situations requiring adaptive and flexible solutions
- Problems with abundant training data or simulation capabilities

### Pros/Cons Summary

| Aspect | Pros | Cons |
|--------|------|------|
| Adaptability | Learns from experience and adapts | Requires extensive training |
| Performance | Can achieve near-optimal performance | Sample inefficient |
| Flexibility | Handles complex, dynamic environments | Black box decisions |
| Scalability | Can scale to large problem instances | Computationally expensive |
| Generalization | Works on unseen test instances | May overfit to training data |

### Key Innovation: Learning-Based Routing

The Deep Reinforcement Learning Scheduler introduces revolutionary innovations:

1. **Experience Replay**: Stores and reuses past experiences for efficient learning
2. **Target Network**: Stabilizes training using separate target network
3. **Epsilon-Greedy Exploration**: Balances exploration and exploitation
4. **Deep Q-Networks**: Uses neural networks to approximate complex value functions
5. **Policy Gradient Methods**: Directly optimizes routing policies through gradient descent

This tier demonstrates how modern artificial intelligence can learn to solve complex routing problems through trial and error, without explicit programming of routing rules. The agent discovers strategies that may be difficult for humans to design, potentially finding novel solutions that outperform traditional optimization methods. This represents the cutting edge of machine learning applications in transportation and logistics optimization.